# Movimientos en masa en roca

Las estructuras geológicas de los materiales que conforman las laderas son una de las principales factores condicionantes de movimientos en masa en roca. Los planos de estratificación, diaclasas o foliación pueden controlar tanto el mecanismo como la actividad de los movimientos en masa. Es por esta razón que analizar la relación entre dichos planos de debilidad con el aspecto y pendiente de las laderas pueden ser clave para entender la ocurrencia. La disponibilidad de modelos digitales del terreno y sistemas de información geográfica ofrecen una excelente posibilidad de caracterizar estas relaciones sobre extensas áreas utilizando los modelos de elevación y datos estructurales.

Como soporte a los modelos con base física para macizos rocosos, las técnicas computacionales actuales permiten la implementación de análisis espaciales que correlacionen información geo-estructural y morfológica de las laderas para evaluar su estabilidad (Grelle et al., 2011). Para este análisis, se definen las relaciones geométricas entre la aptitud de las discontinuidades (rumbo y buzamiento) y la geometría de la ladera (pendiente y aspecto) (Guzzetti et al., 2006; Gunther, 2003; Hammond et al., 1992; Montgomery & Dietrich, 1994; Pack et al., 1998). Estos modelos requieren información espacialmente distribuida sobre la geometría de las laderas y datos estructurales de las discontinuidades para realizar los análisis celda por celda sobre toda el área de estudio (Günther, 2003; Günther et al., 2004; Santagelo et al., 2014). 

Los métodos cinemáticos no toman en consideración las fuerzas que se
ejercen sobre determinado bloque y el peso de este, solo se considera la libertad que tenga la roca de
moverse influenciada solo por su geometría y puntos de apoyo. Los análisis cinemáticos implementados mediante SIG para extensas áreas requieren dominios morfoestructurales, estableciendo áreas geográficas caracterizadas por relaciones geométricas homogéneas entre la actitud de las estructuras geológicas y la geometría de las laderas. Para identificar dichos dominios morfoestructurales autores como Grelle et al. (2011) y Santagelo et al. (2014) han propuesto evaluaciones discretas mediante regla de clasificación o evaluaciones continuas a través de índices que relacionan la geometría de las laderas y la aptitud de las discontinuidades. Finalmente, cada dominio estructural, de acuerdo con la disposición estructural y geometría de la ladera, presenta un tipo principal de cinemática de falla a lo largo de las discontinuidades: fallas planares, fallas en cuña, y fallas por volcamiento. (Cruden & varnes, 1996; Hoek y Bray, 1981).

Uno de los índices utilizados es TOBIA (*TOpographic/Bedding plane Intersection Angle*) propuesto por Meentemeyer & Moody (2000). este indice señala la conformidad entre la ladera y los planos del material de la ladera. Si los planos de la roca buzan en el mismo sentido del aspecto de la ladera, se clasifica como *cataclinal*, si los planos buzan en direccion opuesta, es clasificada como *anaclinal*, y si buza de forma ortogonal es clasificada como *ortoclinal*. Las cataclinales y anaclinales pueden subclasificarse de acuerdo con el ángulo de buzamiento y la pendiente de la ladera, como se presenta en la figura. Las laderas cataclinales (*dip slope* y *overdip sope*) son susceptibles a deslizamientos planares. Mientras que las laderas anaclinales son susceptibles a movimientos en  masa tipo volcamiento y caida. Y las laderas ortoclinales con diferentes familias de diaclasas pueden formar movimientos en masa tipo cuña.

:::{figure-md} roca
<img src="https://i.pinimg.com/564x/e1/b2/c9/e1b2c9845c93e4cb81807adbc02ec8cc.jpg" alt="roca laderas" width="600px">

Clasificación de laderas de acuerdo con el lineamiento entre el aspecto y pendiente con el rumbo y buzamiento de la estructuras. Tomado de {cite:t}`meentemeyer2000`.
:::

En este sentido, y con las 4 siguientes variables pendiente (S 0-90°), aspecto (A 0-360°), buzamiento ($\theta$ 0-90°) y dirección de buzamiento ($\alpha$ 0-360°), puede ser clasificadas todas las celdas que representa el área de interés. Esta clasificación puede realizarse de forma categórica con el valor de *L* :

$L=\sqrt{(cos\alpha - cos A)^2 + (sen\alpha - senA)^2}$

Los rangos de los valores son considerados en este documento $\pm$ 45, sin embargo este valor puede reducirse con creterio de campo, un valor razonable puede ser $\pm$ 20°, como se presenta en las figuras de los análisis cinemáticos.

## Objetivos de aprendizaje

Al finalizar este capítulo, el estudiante será capaz de:

- Clasificar las laderas según la relación entre el buzamiento de la ladera y el buzamiento de las discontinuidades (cataclinales, ortoclinales, anaclinales).
- Aplicar el análisis cinemático para evaluar la posibilidad de falla planar y volcamiento a partir de datos de campo.
- Interpretar los resultados del análisis cinemático en términos de estabilidad de laderas en macizos rocosos.
- Identificar los datos de campo necesarios para un análisis de estabilidad en roca.

**Duración estimada:** 2–3 horas

## Requisitos previos

- **Python:** `numpy`, `matplotlib`.
- **Geología estructural:** sistemas de discontinuidades, buzamiento, dirección de buzamiento, representación en estereored.
- **Geotecnia de roca:** RQD, clasificaciones RMR/Q, resistencia de discontinuidades (JCS, JRC, ángulo de fricción básica).
- **Capítulo anterior:** `06_variables` — variables condicionantes en macizos rocosos.

## Implementación: Análisis Cinemático de Fallas en Roca

El **análisis cinemático** evalúa la posibilidad de que un bloque de roca falle mediante diferentes mecanismos (deslizamiento planar, deslizamiento en cuña, volcamiento) a partir de la orientación de las discontinuidades y de la ladera.

### Falla planar
Ocurre cuando el plano de discontinuidad buza en la misma dirección que la ladera ($|\text{DD}_{\text{disc}} - \text{DD}_{\text{ladera}}| < 20°$) y el buzamiento de la discontinuidad es menor que el de la ladera pero mayor que el ángulo de fricción:

$$\phi < \psi_p < \psi_f$$

donde $\psi_f$ = buzamiento de la ladera, $\psi_p$ = buzamiento del plano, $\phi$ = ángulo de fricción.

### Falla por volcamiento
Ocurre cuando la discontinuidad buza en dirección opuesta a la ladera y el buzamiento es mayor que $(90° - \phi)$:

$$\psi_p > 90° - \phi$$

El siguiente código evalúa ambas condiciones para un conjunto de familias de discontinuidades medidas en campo.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def analisis_cinematico_planar(DD_disc, buz_disc, DD_ladera, buz_ladera, phi,
                               tol_dir=20):
    """
    Análisis cinemático para falla planar.

    Parámetros
    ----------
    DD_disc   : dirección de buzamiento de la discontinuidad [°]
    buz_disc  : buzamiento de la discontinuidad [°]
    DD_ladera : dirección de buzamiento de la ladera [°]
    buz_ladera: buzamiento de la ladera [°]
    phi       : ángulo de fricción [°]
    tol_dir   : tolerancia en dirección [°] (usualmente 20°)

    Retorna
    -------
    bool : True si la falla planar es cinemáticamente posible
    """
    diff_dir = abs(DD_disc - DD_ladera) % 360
    diff_dir = min(diff_dir, 360 - diff_dir)
    cond_dir = diff_dir < tol_dir                  # misma dirección ± 20°
    cond_buz = (buz_disc > phi) and (buz_disc < buz_ladera)  # phi < ψp < ψf
    return cond_dir and cond_buz

def analisis_cinematico_volcamiento(DD_disc, buz_disc, DD_ladera, phi, tol_dir=20):
    """
    Análisis cinemático para falla por volcamiento.

    Parámetros
    ----------
    DD_disc   : dirección de buzamiento de la discontinuidad [°]
    buz_disc  : buzamiento de la discontinuidad [°]
    DD_ladera : dirección de buzamiento de la ladera [°]
    phi       : ángulo de fricción [°]
    tol_dir   : tolerancia en dirección [°]

    Retorna
    -------
    bool : True si el volcamiento es cinemáticamente posible
    """
    # Dirección opuesta a la ladera ± tolerancia
    diff_dir = abs(DD_disc - (DD_ladera + 180) % 360) % 360
    diff_dir = min(diff_dir, 360 - diff_dir)
    cond_dir  = diff_dir < tol_dir
    cond_buz  = buz_disc > (90 - phi)   # ψp > 90° − φ
    return cond_dir and cond_buz

# ── Parámetros de la ladera ───────────────────────────────────────────────────
DD_ladera  = 130   # dirección de buzamiento de la ladera [°]
buz_ladera = 65    # buzamiento de la ladera [°]
phi        = 30    # ángulo de fricción de las discontinuidades [°]

# ── Familias de discontinuidades medidas en campo ─────────────────────────────
familias = [
    {'nombre': 'J1', 'DD': 125, 'buz': 50},
    {'nombre': 'J2', 'DD': 140, 'buz': 70},
    {'nombre': 'J3', 'DD': 310, 'buz': 55},
    {'nombre': 'J4', 'DD': 80,  'buz': 40},
    {'nombre': 'S0', 'DD': 135, 'buz': 30},
]

# ── Evaluación cinemática ─────────────────────────────────────────────────────
print('=' * 60)
print(f'LADERA: DD = {DD_ladera}°, Buz = {buz_ladera}°')
print(f'Ángulo de fricción: φ = {phi}°')
print('=' * 60)
print(f'{"Familia":<8} {"DD":<6} {"Buz":<6} {"Planar":<12} {"Volcamiento"}')
print('-' * 50)

resultados = []
for f in familias:
    planar = analisis_cinematico_planar(
        f['DD'], f['buz'], DD_ladera, buz_ladera, phi)
    volc   = analisis_cinematico_volcamiento(
        f['DD'], f['buz'], DD_ladera, phi)
    resultados.append({'nombre': f['nombre'], 'planar': planar, 'volcamiento': volc})
    p_str = '⚠ POSIBLE' if planar else 'No'
    v_str = '⚠ POSIBLE' if volc   else 'No'
    print(f"{f['nombre']:<8} {f['DD']:<6} {f['buz']:<6} {p_str:<12} {v_str}")

# ── Visualización: diagrama de barras de riesgo por familia ───────────────────
nombres  = [r['nombre'] for r in resultados]
planares = [1 if r['planar'] else 0 for r in resultados]
volc_arr = [1 if r['volcamiento'] else 0 for r in resultados]

x = np.arange(len(nombres))
fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - 0.2, planares, 0.35, label='Falla Planar',
               color=['tomato' if v else 'steelblue' for v in planares],
               edgecolor='k')
bars2 = ax.bar(x + 0.2, volc_arr, 0.35, label='Volcamiento',
               color=['orange' if v else 'steelblue' for v in volc_arr],
               edgecolor='k')
ax.set_xticks(x)
ax.set_xticklabels(nombres, fontsize=11)
ax.set_yticks([0, 1])
ax.set_yticklabels(['No posible', 'Cinemáticamente\nposible'], fontsize=10)
ax.set_xlabel('Familia de discontinuidades', fontsize=12)
ax.set_title(
    f'Análisis Cinemático — Ladera DD={DD_ladera}°, Buz={buz_ladera}°\n'
    f'Ángulo de fricción φ = {phi}°',
    fontsize=13
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## Laderas cataclinales
De forma categórica las laderas cataclinales corresponden cuando la diferencia entre $\alpha$ y A es 0 $\pm$ 45°, lo cual corresponde a $0 \leq L \leq 0.7654$. Adicionalmente, para las laderas cataclinales:

Si $-5° \leq \theta-S \leq 5°$ --> *Dip slope*

Si $0 - S \geq 5°$ --> *underdip slope*

Si $\theta - S < -5°$ --> *overdip slope*

Como parte del análisis cinemático clásico se verifica además que la pendiente sea mayor al ángulo de fricción del plano.

:::{figure-md} precision
<img src="https://i.pinimg.com/564x/8b/6c/d9/8b6cd95caed2bd79b8c01172f83a8611.jpg" alt="modelos" width="600px">

Análisis cinemático para deslizamientos planares.
:::

## Laderas ortoclinales
De forma categórica las laderas ortoclinales corresponden cuando al diferencia entre $\alpha$ y A es 90 $\pm$ 45° o 270 $\pm$ 45, lo cual corresponde a $0.7654\leq L \leq 1.8478$

En este caso para el análisis cinemático la linea de intercepción de los dos planos debe tener una direccion igual a S $\pm$ 20°, y el buzamiento debe ser mayor al ángulo de fricción.

:::{figure-md} precision
<img src="https://i.pinimg.com/564x/00/89/19/008919353afe624b23813743fed36aa7.jpg" alt="modelos" width="600px">

Análisis cinemático para deslizamientos en cuña.
:::

## Laderas anaclinales
De forma categórica las laderas anaclinales corresponden cuando al diferencia entre $\alpha$ y A es 180 $\pm$ 45°, lo cual corresponde a $1.8478 \leq L \leq 2$. Adicionalmente para las laderas anaclinales:

Si $-5° \leq \theta - S \leq 5°$ --> *normal escarpment*

Si $\theta - S > 5°$ --> *Subdued escarpment*

Si $\theta - S < -5°$ --> *steepened escarpment*

En este caso, para el análisis cinemático se debe verificar que $90° - \alpha$ sea menor a la pendiente menos el ángulo de fricción.

:::{figure-md} precision
<img src="https://i.pinimg.com/564x/41/2c/d6/412cd65e9a269c371b354d24533199b2.jpg" alt="modelos" width="600px">

Análisis cinemático para movimientos en masa tipo volcamiento.
:::

## Actividades

1. **Levantamiento de campo:** Para un afloramiento rocoso de tu región, mide 5 familias de discontinuidades con brújula geológica (DD y buzamiento). Aplica el análisis cinemático implementado en este capítulo con el buzamiento y la orientación de la ladera medidos en campo. Reporta cuáles familias son potencialmente inestables.

2. **Variación del ángulo de fricción:** Para la ladera analizada en el código, varía φ entre 20° y 40° y evalúa cómo cambia el número de familias con falla planar posible. ¿A partir de qué φ ninguna familia es potencialmente inestable?

3. **Clasificación RMR:** Investiga la clasificación RMR (Rock Mass Rating) de Bieniawski (1989). Aplica los criterios cualitativos a un macizo rocoso conocido de tu región e infiere el ángulo de fricción promedio de las discontinuidades a partir del valor de RMR.

4. **Cuña de falla:** Investiga el mecanismo de falla en cuña (*wedge failure*) y las condiciones cinemáticas para su ocurrencia. ¿Cómo extenderías la función `analisis_cinematico_planar()` para incluir este modo de falla?

## Referencias
```{bibliography}
:style: plain
:filter: docname in docnames
```